# analysis_framework_contract_v1

目标：验证 `AnalysisSpec` 和 `build_model_regime_comparison_report` 可以作为 `src/analysis` 的稳定 pipeline 入口，把 daily backtest 与 market panel 生成统一 `AnalysisReport`、manifest 和中文 review sidecar。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import AnalysisSpec, MarketRegimeConfig, build_model_regime_comparison_report, write_analysis_report
from src.core import DatasetSnapshot

CARD = Path('research/analysis_framework_contract_v1')
REPORT_DIR = CARD / 'analysis_report'
CARD.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CARD, REPORT_DIR

(PosixPath('research/analysis_framework_contract_v1'),
 PosixPath('research/analysis_framework_contract_v1/analysis_report'))

## 构造 contract 输入

这里使用小样本 daily backtest 与 market panel，只验证框架契约，不代表真实模型效果。

In [2]:
dates = list(pd.date_range('2026-01-01', periods=8, freq='D')) * 2
daily = pd.DataFrame({
    'date': dates,
    'model': ['ridge_mse'] * 8 + ['gcn_mse'] * 8,
    'regime': ['legacy'] * 16,
    'nav': [1.00, 1.02, 1.04, 1.01, 0.99, 1.00, 1.01, 1.03, 1.00, 1.05, 1.08, 1.04, 0.94, 0.97, 0.96, 1.02],
    'benchmark_nav': [1.0, 1.03, 1.04, 1.00, 0.91, 0.94, 0.92, 0.97] * 2,
    'position': [0.3, 0.4, 0.4, 0.2, 0.2, 0.3, 0.2, 0.3] * 2,
    'turnover': [0.0, 0.1, 0.0, 0.2, 0.0, 0.1, 0.1, 0.1] * 2,
})

market = pd.DataFrame({
    'date': pd.date_range('2026-01-01', periods=8, freq='D'),
    'close': [100.0, 104.0, 106.0, 101.0, 90.0, 95.0, 93.0, 98.0],
    'volume': [100, 105, 110, 220, 150, 125, 115, 120],
    'soxx_close': [50.0, 51.0, 51.5, 50.5, 48.0, 49.0, 48.4, 49.2],
    'vix': [17.0, 18.0, 19.0, 22.0, 36.0, 25.0, 24.0, 18.0],
    'trend': [0.04, 0.05, 0.04, 0.00, -0.12, 0.04, -0.04, 0.03],
    'drawdown': [0.0, 0.0, 0.0, -0.02, -0.16, -0.08, -0.10, -0.04],
    'realized_vol': [0.01, 0.015, 0.018, 0.02, 0.06, 0.05, 0.052, 0.018],
    'volume_z': [0.0, 0.2, 0.3, 2.4, 0.7, 0.4, 0.2, 0.1],
})

spec = AnalysisSpec(
    artifact_id='analysis_framework_contract_v1',
    baseline_model='ridge_mse',
    windows={'early': ('2026-01-01', '2026-01-04'), 'stress_window': ('2026-01-05', '2026-01-07')},
    source_type='market_panel',
    market_regime_config=MarketRegimeConfig(
        high_vix_threshold=30.0,
        stress_drawdown_threshold=-0.10,
        uptrend_threshold=0.03,
        downtrend_threshold=-0.03,
        high_volatility_threshold=0.04,
        volume_shock_z_threshold=2.0,
    ),
    summary_note='中文 review：统一 analysis pipeline contract 生成 market regime 模型差异报告。',
)

spec.to_dict()

{'schema_version': 'analysis.spec.v1',
 'artifact_id': 'analysis_framework_contract_v1',
 'baseline_model': 'ridge_mse',
 'windows': {'early': ['2026-01-01', '2026-01-04'],
  'stress_window': ['2026-01-05', '2026-01-07']},
 'market_regime_config': {'trend_window': 21,
  'volatility_window': 21,
  'drawdown_window': 63,
  'volume_window': 20,
  'high_vix_threshold': 30.0,
  'low_vix_threshold': 20.0,
  'stress_drawdown_threshold': -0.1,
  'bear_drawdown_threshold': -0.03,
  'uptrend_threshold': 0.03,
  'downtrend_threshold': -0.03,
  'sideways_trend_abs_threshold': 0.01,
  'high_volatility_threshold': 0.04,
  'low_volatility_threshold': 0.02,
  'volume_shock_z_threshold': 2.0,
  'source': 'market_panel'},
 'freq': 'day',
 'source_type': 'market_panel',
 'summary_note': '中文 review：统一 analysis pipeline contract 生成 market regime 模型差异报告。'}

## 运行统一 pipeline

In [3]:
report = build_model_regime_comparison_report(spec=spec, daily_backtest=daily, market_frame=market)

dataset = DatasetSnapshot(
    name='analysis_framework_contract_v1',
    universe=('SOXL', 'SOXX', 'VIX'),
    start='2026-01-01',
    end='2026-01-08',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=('sample_daily_backtest.csv', 'sample_market_panel.csv'),
)
export = write_analysis_report(report, REPORT_DIR, dataset=dataset)

assert report.summary['status'] == 'analysis_pipeline_completed'
assert report.summary['source_type'] == 'market_panel'
assert report.summary['source_report_status'] == 'market_regime_analysis_completed'
assert 'analysis_overview' in report.summary
assert {'market_regimes', 'market_regime_daily_backtest', 'model_review_briefing', 'window_regime_metrics'}.issubset(report.tables)
assert (REPORT_DIR / 'analysis_manifest.json').exists()

export

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'analysis.report.export.v1',
 'artifact_id': 'analysis_framework_contract_v1',
 'artifact_type': 'analysis_report',
 'paths': {'summary': 'analysis_summary.json',
  'market_regimes': 'market_regimes.csv',
  'market_regime_counts': 'market_regime_counts.csv',
  'market_regime_daily_backtest': 'market_regime_daily_backtest.csv',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'segment_effect_matrix': 'segment_effect_matrix.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'model_regime_window_diagnostics': 'model_regime_window_diagnostics.csv',
  'model_review_briefing': 'model_review_briefing.csv',
  'failure_modes': 'failure_modes.csv',
  'manifest': 'analysis_manifest.json'},
 'tab

## 输出摘要

In [4]:
overview = report.summary['analysis_overview']
briefing = report.tables['model_review_briefing']
window_regime = report.tables['window_regime_metrics']
manifest = json.loads((REPORT_DIR / 'analysis_manifest.json').read_text(encoding='utf-8'))

summary = {
    'schema_version': 'research.analysis_framework_contract_v1',
    'report_dir': str(REPORT_DIR),
    'manifest': str(REPORT_DIR / 'analysis_manifest.json'),
    'analysis_pipeline_schema_version': report.summary['analysis_pipeline_schema_version'],
    'source_type': report.summary['source_type'],
    'table_counts': report.table_counts(),
    'analysis_overview': overview,
    'model_review_briefing_rows': int(len(briefing)),
    'window_regime_rows': int(len(window_regime)),
    'manifest_hash': manifest['manifest_hash'],
    'conclusion': '统一 analysis pipeline 可以从 daily backtest + market panel 生成 manifest-ready 模型差异报告。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

summary

{'schema_version': 'research.analysis_framework_contract_v1',
 'report_dir': 'research/analysis_framework_contract_v1/analysis_report',
 'manifest': 'research/analysis_framework_contract_v1/analysis_report/analysis_manifest.json',
 'analysis_pipeline_schema_version': 'analysis.spec.v1',
 'source_type': 'market_panel',
 'table_counts': {'failure_modes': 3,
  'market_regime_counts': 4,
  'market_regime_daily_backtest': 16,
  'market_regimes': 8,
  'model_comparison': 2,
  'model_effect_deltas': 22,
  'model_regime_window_diagnostics': 2,
  'model_review_briefing': 3,
  'qlib_return_frame': 16,
  'qlib_risk_metrics': 44,
  'regime_metrics': 8,
  'segment_effect_matrix': 11,
  'segment_metrics': 22,
  'window_metrics': 4,
  'window_regime_metrics': 8},
 'analysis_overview': {'baseline_model': 'ridge_mse',
  'dashboard_note': '统一 regime/window 模型差异摘要，供 dashboard 总览和 schedule 通知读取。',
  'decision_hint_counts': {'block_or_rework': 2, 'promote_candidate': 1},
  'largest_relative_return_spread_s

## 中文结论

- 做了什么：验证 `AnalysisSpec` 和 `build_model_regime_comparison_report` 可以把 daily backtest 与 market panel 组合成统一 `AnalysisReport`。
- 核心指标：记录 `model_comparison`、`regime_metrics`、`window_regime_metrics`、`model_review_briefing`、`analysis_overview` 和 `analysis_manifest.json`。
- regime 差异：示例模型在 bull、stress、volume_shock 等分段的相对收益、稳定性和回撤差异进入 briefing 与 failure mode。
- 是否晋升：这是 analysis 框架契约验证，不晋升任何模型。
- 风险：示例数据只用于 contract；真实实验仍需使用 qlib/canonical 数据、真实交易成本和样本量复核。
- 下一步：后续 analysis 扩展先接入 pipeline，再由 dashboard/schedule 读取 manifest。